# 🚑 LaneRush — Emergency Vehicle Detection Model Training

This Google Colab notebook trains a **YOLOv8n** object detection model on your uploaded Emergency Vehicle dataset.

### Complete Workflow:
1. **Check GPU**: Verify T4 GPU acceleration is active.
2. **Install Ultralytics**: Install the latest YOLOv8 engine.
3. **Unzip Dataset**: Auto-detects and extracts your uploaded `.zip` into `dataset/`.
4. **Inspect `data.yaml`**: Verifies paths and classes.
5. **Train YOLOv8n**: 30 epochs with transfer learning (~8–12 mins on T4 GPU).
6. **Evaluate & Generate `metrics.json`**: Computes empirical Precision, Recall, and False Alarm rates.
7. **Download Artifacts**: Downloads `best.pt` and `metrics.json` back to your computer for the LaneRush project.

## Step 1: Check GPU & Install Dependencies
> **Note:** Make sure GPU is enabled under **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
# Verify GPU allocation
!nvidia-smi

# Install Ultralytics YOLO
!pip install -q ultralytics

## Step 2: Unzip Your Uploaded Dataset
1. Click the **Folder icon 📁** on the left panel in Colab.
2. Upload your dataset `.zip` file.
3. Run the cell below to automatically find and unzip it into `dataset/`.

In [ ]:
from pathlib import Path

# Find uploaded .zip file in Colab root
zip_candidates = list(Path('.').glob('*.zip'))

if not zip_candidates:
    raise FileNotFoundError(
        "No .zip file found in Colab root. Please upload your dataset zip in the left sidebar file manager!"
    )

zip_file = zip_candidates[0]
print(f"Found uploaded dataset archive: {zip_file.name}")
print("Extracting into 'dataset/' folder ...")

!mkdir -p dataset
!unzip -q "{zip_file}" -d dataset/
print("Extraction complete!")

## Step 3: Verify and Inspect `data.yaml`
Locates the configuration file and prints the detected classes (e.g., `ambulance`, `emergency_vehicle`).

In [ ]:
from pathlib import Path

yaml_files = list(Path('dataset').glob('**/*.yaml'))

if not yaml_files:
    print("Files in dataset/:")
    !ls -la dataset/
    raise FileNotFoundError("Could not find data.yaml inside dataset/. Please check the directory structure above.")

data_yaml_path = yaml_files[0].resolve()
print(f"Located data configuration: {data_yaml_path}")
print("-" * 50)
print(data_yaml_path.read_text())
print("-" * 50)

## Step 4: Train the YOLOv8n Detection Model
Fine-tunes the pretrained `yolov8n.pt` backbone on your emergency vehicle dataset for 30 epochs.

In [ ]:
from ultralytics import YOLO

# 1. Load pretrained YOLOv8 detection model
model = YOLO('yolov8n.pt')

# 2. Train on your unzipped dataset
results = model.train(
    data=str(data_yaml_path),
    epochs=30,
    imgsz=640,
    batch=16,
    project='ml/runs',
    name='ev_detector',
    seed=42,
    workers=4
)

print("Training complete! Best weights saved to ml/runs/ev_detector/weights/best.pt")

## Step 5: Evaluate Model & Generate `metrics.json`
Evaluates the best checkpoint on the test/val split to extract empirical Precision, Recall, and False Alarm rates.

In [ ]:
import json
from pathlib import Path

from ultralytics import YOLO

best_weights = 'ml/runs/ev_detector/weights/best.pt'
model = YOLO(best_weights)

# Evaluate on dataset validation/test split
metrics = model.val(data=str(data_yaml_path))

# Extract bounding box metrics
precision = float(metrics.box.mp)      # Mean precision
recall = float(metrics.box.mr)         # Mean recall (ev_recall)
map50 = float(metrics.box.map50)       # mAP @ IoU 0.50
map50_95 = float(metrics.box.map)      # mAP @ IoU 0.50:0.95

# False alarm rate approximation: 1 - precision
false_alarm = max(0.0, round(1.0 - precision, 4))

eval_results = {
    "model": "YOLOv8n-Detection",
    "ev_recall": round(recall, 4),
    "ev_precision": round(precision, 4),
    "ev_false_alarm": false_alarm,
    "mAP50": round(map50, 4),
    "mAP50_95": round(map50_95, 4)
}

# Save to ml/results/metrics.json
out_dir = Path('ml/results')
out_dir.mkdir(parents=True, exist_ok=True)
metrics_file = out_dir / 'metrics.json'
metrics_file.write_text(json.dumps(eval_results, indent=2))

print("Evaluation Summary:")
print(json.dumps(eval_results, indent=2))

## Step 6: Download the Output Files to Your Computer
Run this cell to download `best.pt` and `metrics.json` directly through your browser.

In [ ]:
from google.colab import files

print("Triggering browser download for metrics.json and best.pt ...")
files.download('ml/results/metrics.json')
files.download('ml/runs/ev_detector/weights/best.pt')

## Step 7: How to Use These Files in Your LaneRush Project

Once the two files finish downloading to your computer:

1. **Move `metrics.json`** into:
   `laneRush/ml/results/metrics.json`

2. **Move `best.pt`** into:
   `laneRush/ml/runs/ev_detector/weights/best.pt`

3. **Update your simulation config** (`configs/exp_main.yaml`):
   Set the empirical `ev_recall` and `ev_false_alarm` values from your `metrics.json`:
   ```yaml
   perception:
     provider: noisy
     noisy:
       ev_recall: <YOUR_MEASURED_RECALL>        # e.g., 0.94
       ev_false_alarm: <YOUR_FALSE_ALARM_RATE>  # e.g., 0.02
       confirmation_k: 2
       confirmation_n: 3
   ```